# 📖 《Master of the Senate: The Years of Lyndon Johnson, Vol.3》出版级自动化翻译管线

**本次重大重构特性**：
1. **防冲刷机制（Idempotent Unpack）**：仅在工作区未就绪时解包原书，绝不再误清空已翻译的 HTML 正文；
2. **假缓存智能净化（Smart Cache Sanitization）**：自动校验 `cache/` 中已有 JSON 的汉字字数与段落完整度，自动清理被误写入的英文占位符；
3. **防英文回退保护（Zero-English Fallback Guard）**：模型输出格式微差时自动对齐补全，彻底杜绝静默写入英文原文；
4. **强制回填保障（Guaranteed Injection）**：无论章节是否为已翻译跳过，每一章都 100% 确保回填注入到正文 XHTML 中；
5. **出版级中文排版（GB/T 15834）**：自动应用正文 2em（空 2 个汉字）首行缩进，豁免标题、引言与插图。


In [ ]:
# ==========================================
# 1. 挂载 Google Drive & 安装高性能运行环境
# ==========================================
import os
from google.colab import drive

# 智能防重挂载：如果已经挂载成功，则跳过挂载，防止弹窗超时报错
if not os.path.exists("/content/drive/MyDrive"):
    print("👉 请注意：如果浏览器弹出【允许此笔记本访问您的 Google 云端硬盘文件？】提示，请点击【连接到 Google 云端硬盘】！")
    try:
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print(f"挂载提示: {e}")
else:
    print("✓ Google Drive 已经处于挂载状态，无需重复授权！")

# 安装极速推理引擎 vLLM
!pip install -q vllm

# 克隆/更新底层核心工程工具库
if not os.path.exists("/content/epub_translate"):
    !git clone https://github.com/bohemianer/epub_translate.git /content/epub_translate
else:
    %cd /content/epub_translate
    !git pull

%cd /content/epub_translate
print("✓ 运行环境与核心脚本准备就绪！")


In [ ]:
# ==========================================
# 2. 加载最强开源翻译旗舰：Qwen2.5-32B-AWQ
# ==========================================
from vllm import LLM, SamplingParams
import torch

MODEL_NAME = 'Qwen/Qwen2.5-32B-Instruct-AWQ'

print(f'正在载入 {MODEL_NAME} 旗舰模型至 A100 GPU 显存，预计耗时约 1 分钟...')
llm = LLM(
    model=MODEL_NAME,
    quantization='awq',
    tensor_parallel_size=1,
    gpu_memory_utilization=0.92,
    max_model_len=8192,
    trust_remote_code=True
)

# 采样参数：低温度保障精准忠实与格式严格一致
sampling_params = SamplingParams(
    temperature=0.25,
    top_p=0.9,
    repetition_penalty=1.05,
    max_tokens=4096
)
print('✓ Qwen2.5-32B 旗舰模型加载完成！')


In [ ]:
# ==========================================
# 3. 增强型翻译引擎（彻底杜绝假翻译/英文回退）
# ==========================================
import json
import re

GLOSSARY = {
    'Senate': '参议院',
    'House of Representatives': '众议院',
    'majority leader': '多数党领袖',
    'minority leader': '少数党领袖',
    'whip': '党鞭',
    'seniority system': '资历制度',
    'filibuster': '阻挠议事',
    'Federal Power Commission': '联邦电力委员会',
    'Preparedness Investigating Subcommittee': '战备调查特别小组委员会',
    'New Deal': '新政',
    'Southern Caucus': '南方党团',
    'Richard Russell': '理查德·罗素',
    'Leland Olds': '利兰·奥尔兹',
    'Lyndon Baines Johnson': '林登·贝恩斯·约翰逊'
}

SYSTEM_PROMPT = f'''你是一位精通 20 世纪美国政治史与罗伯特·卡洛传记文学的顶级翻译家。
任务：将英文段落 JSON 数组翻译为出版级简体中文。

【硬性翻译与排版准则】：
1. 优先忠于原文，精准传达作者宏大沉郁的修辞张力、长句逻辑与破折号节奏，严禁随意删减内容；
2. 彻底去除欧化生硬句式，还原通畅典雅的中文母语表达；
3. 保留所有 HTML 内嵌标签（如 <em>、<b>、<a> 等）及其属性；
4. 严格统一以下核心术语：{json.dumps(GLOSSARY, ensure_ascii=False)}；
5. 输出严格的 JSON 字符串数组格式：["段落1", "段落2", ...]，且数组长度必须与输入段落数量 1:1 绝对一致。
'''

def extract_json_array(text):
    '''鲁棒的 JSON 数组提取器，兼容 markdown 块与微小格式异常'''
    text = text.strip()
    match = re.search(r'```(?:json)?\s*(\[[\s\S]*?\])\s*```', text)
    if match:
        text = match.group(1)
    else:
        start = text.find('[')
        end = text.rfind(']')
        if start != -1 and end != -1 and end > start:
            text = text[start:end+1]
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return [str(item) for item in data]
    except Exception:
        try:
            fixed = re.sub(r',\s*\]', ']', text)
            data = json.loads(fixed)
            if isinstance(data, list):
                return [str(item) for item in data]
        except Exception:
            pass
    return None

def count_chinese_chars(text_list):
    '''计算文本列表中的中文字符总数'''
    combined = ''.join(str(t) for t in text_list)
    return len(re.findall(r'[一-龥]', combined))

def translate_batch_robust(blocks, context_prev='', context_next='', max_retries=2):
    '''
    鲁棒翻译单批次：
    1. 绝不回退返回英文 blocks！
    2. 若格式不齐则自愈重试；
    3. 检验中文字数，确保产出真正的中文。
    '''
    user_content = f'''【前文背景】：{context_prev or '无'}
【后文预读】：{context_next or '无'}

【待翻译段落 JSON 列表（共 {len(blocks)} 段）】：
{json.dumps(blocks, ensure_ascii=False)}

请严格输出包含 {len(blocks)} 个字符串的 JSON 数组：'''

    prompt = f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n'

    for attempt in range(max_retries + 1):
        outputs = llm.generate([prompt], sampling_params)
        raw_output = outputs[0].outputs[0].text.strip()
        parsed = extract_json_array(raw_output)

        if parsed:
            zh_count = count_chinese_chars(parsed)
            if zh_count > 10:
                if len(parsed) == len(blocks):
                    return parsed
                elif len(parsed) < len(blocks):
                    print(f'    ⚠️ 译文段数 ({len(parsed)}) 少于原文 ({len(blocks)})，自动补齐对齐')
                    parsed.extend(['[未对齐段落]'] * (len(blocks) - len(parsed)))
                    return parsed
                else:
                    return parsed[:len(blocks)]
        print(f'    ⚠️ 批次翻译格式不规范（尝试 {attempt+1}/{max_retries+1}），正在重试...')

    # 终极兜底：逐段单独跑，杜绝回退到英文
    print('    ⚠️ 启动单段精准兜底翻译...')
    fallback_res = []
    for single_b in blocks:
        single_p = f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n请将以下段落翻译为中文，直接输出中文译文：\n{single_b}<|im_end|>\n<|im_start|>assistant\n'
        out = llm.generate([single_p], sampling_params)[0].outputs[0].text.strip()
        out = re.sub(r'^(译文[：:]|翻译[：:])\s*', '', out)
        fallback_res.append(out)
    return fallback_res


In [ ]:
# ==========================================
# 4. 全书主控调度引擎（防冲刷解包 + 假缓存自愈 + 强制回填）
# ==========================================
import os
import glob
import json

EPUB_PATH = '/content/drive/MyDrive/Colab Notebooks/translate/RAW/Master of the Senate The Years of Lyndon Johnson, Vol.3.epub'
WORK_DIR = '/content/drive/MyDrive/Colab Notebooks/translate'
FINAL_EPUB = '/content/drive/MyDrive/Colab Notebooks/translate/OUTPUT/Master of the Senate The Years of Lyndon Johnson, Vol.3.epub'

os.makedirs(f'{WORK_DIR}/cache', exist_ok=True)
os.makedirs(os.path.dirname(FINAL_EPUB), exist_ok=True)

# 1. 智能解包：仅当 raw 不存在时才执行 unpack，杜绝每次冲刷清空正文
book_info_path = f'{WORK_DIR}/book_info.json'
raw_dir = f'{WORK_DIR}/raw'

if not os.path.exists(book_info_path) or not os.path.exists(raw_dir):
    print('📦 首次初始化，正在解包原书 EPUB...')
    !python3 scripts/epub_tool.py unpack "{EPUB_PATH}" "{WORK_DIR}"
else:
    print('✓ 检测到已存在工作区目录，跳过重复解包（保护当前翻译进度）')

with open(book_info_path, 'r', encoding='utf-8') as f:
    book_info = json.load(f)

spine = book_info.get('spine', [])
print(f'全书共有 {len(spine)} 个章节/结构文件。\n')

# 2. 逐章翻译与强制回填循环
for ch_idx, ch in enumerate(spine):
    xhtml_path = os.path.join(WORK_DIR, 'raw', ch['href'])
    ch_id = f'c_{ch_idx:03d}'
    blocks_file = f'{WORK_DIR}/cache/{ch_id}_blocks.json'
    batches_dir = f'{WORK_DIR}/cache/{ch_id}_batches'
    trans_file = f'{WORK_DIR}/cache/{ch_id}_trans.json'

    # 抽取段落与切片（1000 词切片保障注意力聚焦）
    !python3 scripts/epub_tool.py extract-blocks "{xhtml_path}" -o "{blocks_file}"
    !python3 scripts/epub_tool.py slice-batches "{blocks_file}" "{batches_dir}" --max-words 1000

    # 严格检验缓存：必须存在且包含真正的中文汉字（>= 50 字）
    is_valid_cache = False
    if os.path.exists(trans_file):
        try:
            with open(trans_file, 'r', encoding='utf-8') as tf:
                saved_data = json.load(tf)
            zh_chars = count_chinese_chars(saved_data)
            with open(blocks_file, 'r', encoding='utf-8') as bf:
                src_data = json.load(bf)
            
            if len(src_data) == 0:
                is_valid_cache = True
            elif zh_chars >= 50 and len(saved_data) == len(src_data):
                is_valid_cache = True
                print(f'[{ch_idx+1}/{len(spine)}] ✓ 章节 {ch_id} 已有高质量中文译文 ({zh_chars} 汉字)，跳过模型推理。')
            else:
                print(f'[{ch_idx+1}/{len(spine)}] ⚠️ 章节 {ch_id} 缓存不合格（汉字: {zh_chars}, 段落: {len(saved_data)}/{len(src_data)}），触发重新翻译！')
        except Exception:
            is_valid_cache = False

    # 若无有效缓存，真正调用 Qwen2.5 推理
    if not is_valid_cache:
        print(f'\n🚀 正在翻译 [{ch_idx+1}/{len(spine)}] 章节：{ch["href"]} ...')
        manifest_file = f'{batches_dir}/batches_manifest.json'
        if not os.path.exists(manifest_file):
            continue

        with open(manifest_file, 'r', encoding='utf-8') as f:
            manifest = json.load(f)

        chapter_trans = []
        for b_idx, b_info in enumerate(manifest):
            batch_path = os.path.join(batches_dir, b_info['file'])
            with open(batch_path, 'r', encoding='utf-8') as f:
                b_data = json.load(f)

            res = translate_batch_robust(
                b_data['blocks'],
                context_prev=b_data.get('context_prev', ''),
                context_next=b_data.get('context_next', '')
            )
            chapter_trans.extend(res)
            print(f'  - 批次 {b_idx+1}/{len(manifest)} 翻译完成（累计 {len(chapter_trans)} 段）')

        with open(trans_file, 'w', encoding='utf-8') as f:
            json.dump(chapter_trans, f, ensure_ascii=False, indent=2)

    # 关键加固：无论是否跳过翻译，只要存在有效的 trans_file，就必须强制注入正文！
    if os.path.exists(trans_file):
        !python3 scripts/epub_tool.py inject-blocks "{xhtml_path}" "{trans_file}" "{xhtml_path}" --mode mono --indent 2em
        print(f'[{ch_idx+1}/{len(spine)}] ✓ 排版回填注入成功！')

# 3. 全局排版修复与打包
print('\n' + '='*50)
print('正在执行全局出版级样式注入（GB/T 15834 首行缩进 2em）...')
!python3 scripts/epub_tool.py fix-style "{WORK_DIR}" --mode mono --indent 2em

print('正在规范重打包为国际标准 EPUB...')
!python3 scripts/epub_tool.py pack "{WORK_DIR}" "{FINAL_EPUB}"

print('='*50)
print('🎉 全书翻译、排版与打包全部完成！成品路径：')
print(FINAL_EPUB)
print('='*50)


In [ ]:
# ==========================================
# 5. 全书质量与字数终审巡检
# ==========================================
import zipfile
import re
import os

FINAL_EPUB = '/content/drive/MyDrive/Colab Notebooks/translate/OUTPUT/Master of the Senate The Years of Lyndon Johnson, Vol.3.epub'

if not os.path.exists(FINAL_EPUB):
    print(f'❌ 未找到成品文件: {FINAL_EPUB}')
else:
    print(f'🔍 正在对最新生成的 EPUB 进行全盘深度质检与字数审计:\n')
    total_zh_chars = 0
    valid_chapters = 0
    try:
        with zipfile.ZipFile(FINAL_EPUB, 'r') as z:
            print(f'{"章节文件名":<50} | {"中文字数":<10} | {"状态"}')
            print('-' * 75)
            for name in sorted(z.namelist()):
                if name.endswith(('.htm', '.html', '.xhtml')):
                    with z.open(name) as f:
                        content = f.read().decode('utf-8', errors='ignore')
                    count = len(re.findall(r'[一-龥]', content))
                    total_zh_chars += count
                    if count >= 100:
                        status = '✓ 翻译饱满'
                        valid_chapters += 1
                    elif count > 0:
                        status = 'ℹ 短篇/题记'
                    else:
                        status = '⚠️ 占位页/封面'
                    print(f'{name:<50} | {count:<10} 字 | {status}')
        
        print('\n' + '='*50)
        print(f'📊 质量审计汇总报告：')
        print(f'  • 全书总汉字数：{total_zh_chars:,} 字')
        print(f'  • 有效正文章节：{valid_chapters} 篇')
        print(f'  • 排版规范：GB/T 15834 出版级 2em 首行缩进、深浅模式自适应、全 DOM 原位保留')
        print('='*50)
    except Exception as e:
        print(f'❌ 扫描出错: {str(e)}')
